In [1]:
!apt-get update -q
!apt-get install -y postgresql postgresql-contrib libpq-dev python3-dev build-essential -q

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,921 kB]
Get:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,810 kB]
Get:14 http://

In [2]:
!service postgresql start


 * Starting PostgreSQL 14 database server
   ...done.


In [3]:
!sudo -u postgres psql -c "ALTER USER postgres PASSWORD 'postgres';"
!sudo -u postgres psql -c "CREATE DATABASE IF NOT EXISTS milestone1_users;" 2>/dev/null || sudo -u postgres psql -c "SELECT 1 FROM pg_database WHERE datname='milestone1_users'" | grep -q 1 || sudo -u postgres psql -c "CREATE DATABASE milestone1_users;"


ALTER ROLE
CREATE DATABASE


In [4]:
import subprocess
result = subprocess.run(['sudo', '-u', 'postgres', 'psql', '-lqt'], capture_output=True, text=True)
if 'milestone1_users' not in result.stdout:
    subprocess.run(['sudo', '-u', 'postgres', 'psql', '-c', 'CREATE DATABASE milestone1_users;'])
    print('Database created.')
else:
    print('Database already exists.')

Database already exists.


In [5]:
import psycopg2
conn = psycopg2.connect(dbname='milestone1_users', user='postgres', password='postgres', host='localhost')
print('✅ PostgreSQL Connected Successfully!')
conn.close()

✅ PostgreSQL Connected Successfully!


In [6]:
!pip install streamlit psycopg2-binary bcrypt pyjwt watchdog pyngrok -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 104.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 101.0 MB/s eta 0:00:00


In [7]:
!pip install streamlit pyjwt bcrypt python-dotenv pyngrok nltk streamlit-option-menu plotly textstat PyPDF2 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.3/829.3 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 45.3 MB/s eta 0:00:00


In [8]:
%%writefile text_readability.py
import textstat

class ReadabilityAnalyzer:
    def __init__(self, text):
        self.text = text
        self.num_sentences = textstat.sentence_count(text)
        self.num_words = textstat.lexicon_count(text, removepunct=True)
        self.num_syllables = textstat.syllable_count(text)
        self.complex_words = textstat.difficult_words(text)
        self.char_count = textstat.char_count(text)

    def get_all_metrics(self):
        return {
            "Flesch Reading Ease": textstat.flesch_reading_ease(self.text),
            "Flesch-Kincaid Grade": textstat.flesch_kincaid_grade(self.text),
            "SMOG Index": textstat.smog_index(self.text),
            "Gunning Fog": textstat.gunning_fog(self.text),
            "Coleman-Liau": textstat.coleman_liau_index(self.text)
        }


Writing text_readability.py


In [9]:
!pip install plotly PyPDF2 textstat py-readability-metrics

In [10]:
%%writefile app.py
import streamlit as st
import psycopg2
import jwt
import datetime
import bcrypt
import os
import re
import time
import smtplib
import secrets
import hmac
import hashlib
import struct
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from streamlit_option_menu import option_menu
import plotly.graph_objects as go
import PyPDF2
import textstat
from text_readability import ReadabilityAnalyzer

# ==============================
# CONFIGURATION
# ==============================

EMAIL_PASSWORD = os.getenv("EMAIL_PASSWORD")
SECRET_KEY = os.getenv("JWT_SECRET", "dev_secret_key")
EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS", "springboardmentor018@gmail.com")
ALGORITHM = "HS256"
ACCESS_TOKEN_EXPIRE_MINUTES = 30
OTP_EXPIRY_MINUTES = 10
MAX_LOGIN_ATTEMPTS = 3
LOCKOUT_SECONDS = 300  # 5 minutes

DB_NAME = os.getenv("DB_NAME", "milestone1_users")
DB_USER = os.getenv("DB_USER", "postgres")
DB_PASSWORD = os.getenv("DB_PASSWORD", "postgres")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5432")

# ==============================
# PAGE CONFIG & CUSTOM STYLING
# ==============================

st.set_page_config(page_title="Secure Auth System", page_icon="🔐", layout="wide", initial_sidebar_state="expanded")

st.markdown("""
<style>
#MainMenu {visibility: hidden;}
footer {visibility: hidden;}
/* Do NOT hide header - it contains the sidebar toggle button */

.stApp {
    background-color: #0E1117;
    color: #ffffff;
}

h1, h2, h3 {
    color: #bf00ff !important;
    font-family: 'Courier New', monospace;
    text-shadow: 0 0 10px #bf00ff;
    text-align: center;
}

.stTextInput > div > div > input {
    background-color: #1f2937;
    color: #bf00ff;
    border: 1px solid #374151;
    border-radius: 5px;
}

.stTextInput > div > div > input:focus {
    border-color: #bf00ff;
    box-shadow: 0 0 5px #bf00ff;
}

.stSelectbox > div > div {
    background-color: #1f2937;
    color: #bf00ff;
    border: 1px solid #374151;
}

.stButton > button {
    width: 100%;
    border-radius: 10px;
    height: 45px;
    background-color: #1f2937;
    color: #bf00ff;
    font-weight: bold;
    border: 1px solid #bf00ff;
    font-family: 'Courier New', monospace;
    transition: all 0.3s ease;
}

.stButton > button:hover {
    background-color: #bf00ff;
    color: #0e1117;
    box-shadow: 0 0 15px #bf00ff;
}

.strength-weak { color: #ff4b4b; font-weight: bold; }
.strength-medium { color: #ffa500; font-weight: bold; }
.strength-strong { color: #bf00ff; font-weight: bold; }

.stTextArea > div > div > textarea {
    background-color: #1f2937;
    color: #bf00ff;
    border: 1px solid #374151;
    border-radius: 5px;
}
.stTextArea > div > div > textarea:focus {
    border-color: #bf00ff;
    box-shadow: 0 0 5px #bf00ff;
}
section[data-testid="stSidebar"] {
    background-color: #1a1c24;
}
.stTabs [data-baseweb="tab"] {
    background-color: #1f2937;
    color: #9ca3af;
    border-radius: 5px;
    padding: 8px 16px;
}
.stTabs [aria-selected="true"] {
    background-color: #bf00ff !important;
    color: #0e1117 !important;
}
[data-testid="stMetricValue"] {
    color: #bf00ff;
}
</style>
""", unsafe_allow_html=True)

# ==============================
# DATABASE
# ==============================

def get_connection():
    return psycopg2.connect(
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        host=DB_HOST,
        port=DB_PORT
    )

def create_tables():
    conn = get_connection()
    cur = conn.cursor()

    # Users table with security question
    cur.execute("""
        CREATE TABLE IF NOT EXISTS users (
            id SERIAL PRIMARY KEY,
            username VARCHAR(100) UNIQUE NOT NULL,
            email VARCHAR(150) UNIQUE NOT NULL,
            password TEXT NOT NULL,
            security_question TEXT NOT NULL,
            security_answer TEXT NOT NULL,
            created_at TIMESTAMP DEFAULT NOW()
        );
    """)

    # Password history table (prevents reuse)
    cur.execute("""
        CREATE TABLE IF NOT EXISTS password_history (
            id SERIAL PRIMARY KEY,
            email VARCHAR(150) NOT NULL,
            password TEXT NOT NULL,
            set_at TIMESTAMP DEFAULT NOW(),
            FOREIGN KEY (email) REFERENCES users(email) ON DELETE CASCADE
        );
    """)

    # Login attempts table (rate limiting)
    cur.execute("""
        CREATE TABLE IF NOT EXISTS login_attempts (
            email VARCHAR(150) PRIMARY KEY,
            attempts INTEGER DEFAULT 0,
            last_attempt DOUBLE PRECISION DEFAULT 0
        );
    """)

    conn.commit()
    cur.close()
    conn.close()

create_tables()

# ==============================
# RATE LIMITING
# ==============================

def get_login_attempts(email):
    conn = get_connection()
    cur = conn.cursor()
    cur.execute("SELECT attempts, last_attempt FROM login_attempts WHERE email=%s", (email,))
    data = cur.fetchone()
    cur.close()
    conn.close()
    return data if data else (0, 0)

def increment_login_attempts(email):
    conn = get_connection()
    cur = conn.cursor()
    attempts, _ = get_login_attempts(email)
    now = time.time()
    cur.execute("""
        INSERT INTO login_attempts (email, attempts, last_attempt)
        VALUES (%s, %s, %s)
        ON CONFLICT (email) DO UPDATE SET attempts=%s, last_attempt=%s
    """, (email, attempts + 1, now, attempts + 1, now))
    conn.commit()
    cur.close()
    conn.close()

def reset_login_attempts(email):
    conn = get_connection()
    cur = conn.cursor()
    cur.execute("DELETE FROM login_attempts WHERE email=%s", (email,))
    conn.commit()
    cur.close()
    conn.close()

def is_rate_limited(email):
    attempts, last_attempt = get_login_attempts(email)
    if attempts >= MAX_LOGIN_ATTEMPTS:
        elapsed = time.time() - last_attempt
        if elapsed < LOCKOUT_SECONDS:
            return True, int(LOCKOUT_SECONDS - elapsed)
        else:
            reset_login_attempts(email)
    return False, 0

# ==============================
# USER MANAGEMENT
# ==============================

def check_user_exists_by_email(email):
    conn = get_connection()
    cur = conn.cursor()
    cur.execute("SELECT id FROM users WHERE email=%s", (email,))
    data = cur.fetchone()
    cur.close()
    conn.close()
    return data is not None

def check_user_exists_by_username(username):
    conn = get_connection()
    cur = conn.cursor()
    cur.execute("SELECT id FROM users WHERE username=%s", (username,))
    data = cur.fetchone()
    cur.close()
    conn.close()
    return data is not None

def register_user(username, email, password, security_question, security_answer):
    conn = get_connection()
    cur = conn.cursor()
    try:
        hashed_pw = bcrypt.hashpw(password.encode(), bcrypt.gensalt()).decode()
        cur.execute("""
            INSERT INTO users (username, email, password, security_question, security_answer)
            VALUES (%s, %s, %s, %s, %s)
        """, (username, email, hashed_pw, security_question, security_answer.strip().lower()))
        # Save to password history
        cur.execute("""
            INSERT INTO password_history (email, password) VALUES (%s, %s)
        """, (email, hashed_pw))
        conn.commit()
        return True, "Success"
    except Exception as e:
        conn.rollback()
        return False, str(e)
    finally:
        cur.close()
        conn.close()

def authenticate_user(email, password):
    conn = get_connection()
    cur = conn.cursor()
    cur.execute("SELECT username, password FROM users WHERE email=%s", (email,))
    result = cur.fetchone()
    cur.close()
    conn.close()
    if result:
        username_db, hashed_pw = result
        if bcrypt.checkpw(password.encode(), hashed_pw.encode()):
            reset_login_attempts(email)
            return True, username_db
    increment_login_attempts(email)
    return False, None

def check_password_reused(email, new_password):
    conn = get_connection()
    cur = conn.cursor()
    cur.execute("SELECT password FROM password_history WHERE email=%s ORDER BY set_at DESC", (email,))
    history = cur.fetchall()
    cur.close()
    conn.close()
    for (stored_hash,) in history:
        if bcrypt.checkpw(new_password.encode(), stored_hash.encode()):
            return True
    return False

def check_is_old_password(email, password):
    conn = get_connection()
    cur = conn.cursor()
    cur.execute("SELECT password, set_at FROM password_history WHERE email=%s ORDER BY set_at DESC", (email,))
    history = cur.fetchall()
    cur.close()
    conn.close()
    for stored_hash, set_at in history:
        if bcrypt.checkpw(password.encode(), stored_hash.encode()):
            return set_at
    return None

def update_password(email, new_password):
    conn = get_connection()
    cur = conn.cursor()
    hashed_pw = bcrypt.hashpw(new_password.encode(), bcrypt.gensalt()).decode()
    cur.execute("UPDATE users SET password=%s WHERE email=%s", (hashed_pw, email))
    cur.execute("INSERT INTO password_history (email, password) VALUES (%s, %s)", (email, hashed_pw))
    conn.commit()
    cur.close()
    conn.close()

def get_security_question(email):
    conn = get_connection()
    cur = conn.cursor()
    cur.execute("SELECT security_question, security_answer FROM users WHERE email=%s", (email,))
    result = cur.fetchone()
    cur.close()
    conn.close()
    return result

def get_all_users():
    conn = get_connection()
    cur = conn.cursor()
    cur.execute("SELECT username, email, created_at FROM users ORDER BY created_at DESC")
    data = cur.fetchall()
    cur.close()
    conn.close()
    return data

def delete_user(email):
    conn = get_connection()
    cur = conn.cursor()
    cur.execute("DELETE FROM users WHERE email=%s", (email,))
    conn.commit()
    cur.close()
    conn.close()

# ==============================
# JWT
# ==============================

def create_access_token(data: dict, expires_minutes=ACCESS_TOKEN_EXPIRE_MINUTES):
    to_encode = data.copy()
    expire = datetime.datetime.utcnow() + datetime.timedelta(minutes=expires_minutes)
    to_encode.update({"exp": expire})
    return jwt.encode(to_encode, SECRET_KEY, algorithm=ALGORITHM)

def verify_token(token):
    try:
        return jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])
    except:
        return None

# ==============================
# VALIDATION
# ==============================

def is_valid_email(email):
    pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
    return re.match(pattern, email) is not None

def check_password_strength(password):
    has_upper = bool(re.search(r"[A-Z]", password))
    has_lower = bool(re.search(r"[a-z]", password))
    has_digit = bool(re.search(r"\d", password))
    has_special = bool(re.search(r"[!@#$%^&*(),.?\":{}|<>]", password))
    has_space = bool(re.search(r"\s", password))

    if has_space:
        return "Weak", ["No spaces allowed"]
    is_alphanum = (has_upper or has_lower) and has_digit

    if len(password) >= 8 and is_alphanum and has_special:
        return "Strong", []
    if len(password) >= 8 and is_alphanum:
        return "Medium", ["Add special characters for Strong"]
    return "Weak", ["Min 8 chars with letters and numbers required"]

def get_relative_time(dt):
    if not dt:
        return "some time ago"
    try:
        if isinstance(dt, str):
            dt = datetime.datetime.strptime(dt, "%Y-%m-%d %H:%M:%S")
        diff = datetime.datetime.utcnow() - dt
        days = diff.days
        if days > 365: return f"{days // 365} year(s) ago"
        elif days > 30: return f"{days // 30} month(s) ago"
        elif days > 0: return f"{days} day(s) ago"
        else: return "recently"
    except:
        return str(dt)

# ==============================
# OTP
# ==============================

def generate_otp():
    secret = secrets.token_bytes(20)
    counter = int(time.time())
    msg = struct.pack(">Q", counter)
    hmac_hash = hmac.new(secret, msg, hashlib.sha1).digest()
    offset = hmac_hash[19] & 0xf
    code = (
        (hmac_hash[offset] & 0x7f) << 24 |
        (hmac_hash[offset + 1] & 0xff) << 16 |
        (hmac_hash[offset + 2] & 0xff) << 8 |
        (hmac_hash[offset + 3] & 0xff)
    )
    return f"{code % 1000000:06d}"

def create_otp_token(otp, email):
    otp_hash = bcrypt.hashpw(otp.encode(), bcrypt.gensalt()).decode()
    payload = {
        'otp_hash': otp_hash,
        'sub': email,
        'type': 'password_reset',
        'iat': datetime.datetime.utcnow(),
        'exp': datetime.datetime.utcnow() + datetime.timedelta(minutes=OTP_EXPIRY_MINUTES)
    }
    return jwt.encode(payload, SECRET_KEY, algorithm=ALGORITHM)

def verify_otp_token(token, input_otp, email):
    try:
        payload = jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])
        if payload.get('sub') != email:
            return False, "Token mismatch"
        if bcrypt.checkpw(input_otp.encode(), payload['otp_hash'].encode()):
            return True, "Valid"
        return False, "Invalid OTP"
    except Exception as e:
        return False, str(e)

def send_otp_email(to_email, otp):
    msg = MIMEMultipart()
    msg['From'] = f"Secure Auth <{EMAIL_ADDRESS}>"
    msg['To'] = to_email
    msg['Subject'] = "🔐 Password Reset OTP"
    body = f"""
    <html><body style="background:#0e1117;font-family:monospace;padding:40px;text-align:center;">
    <div style="background:#1f2937;border-radius:12px;padding:40px;max-width:500px;margin:auto;border:1px solid #374151;">
        <h2 style="color:#bf00ff;text-shadow:0 0 5px #bf00ff;">🔐 Password Reset OTP</h2>
        <p style="color:#9ca3af;">Use the OTP below to reset your password for <span style="color:#bf00ff;">{to_email}</span>.</p>
        <div style="background:#0e1117;color:#bf00ff;font-size:32px;font-weight:700;letter-spacing:8px;padding:20px;border-radius:8px;margin:30px 0;border:1px solid #bf00ff;">
            {otp}
        </div>
        <p style="color:#9ca3af;">Valid for <strong>{OTP_EXPIRY_MINUTES} minutes</strong>. Do not share this code.</p>
        <p style="color:#6b7280;font-size:12px;">&copy; 2026 Secure Auth System</p>
    </div></body></html>
    """
    msg.attach(MIMEText(body, 'html'))
    try:
        s = smtplib.SMTP('smtp.gmail.com', 587)
        s.starttls()
        s.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
        s.sendmail(EMAIL_ADDRESS, to_email, msg.as_string())
        s.quit()
        return True, "Sent"
    except Exception as e:
        return False, str(e)

# ==============================
# SESSION INIT
# ==============================

SECURITY_QUESTIONS = [
    "What is your pet's name?",
    "What is your mother's maiden name?",
    "Who was your favorite teacher?",
    "What city were you born in?",
    "What was the name of your first school?"
]

for key, val in [("jwt_token", None), ("page", "login"), ("username", None)]:
    if key not in st.session_state:
        st.session_state[key] = val

# ==============================
# SIGNUP PAGE
# ==============================

def signup_page():
    col1, col2, col3 = st.columns([1, 2, 1])
    with col2:
        st.title("🔐 Create Account")

        username = st.text_input("Username")
        email = st.text_input("Email")
        password = st.text_input("Password", type="password")

        if password:
            strength, feedback = check_password_strength(password)
            color = {"Strong": "#bf00ff", "Medium": "#ffa500", "Weak": "#ff4b4b"}[strength]
            st.markdown(f"Strength: <span style='color:{color};font-weight:bold;'>{strength}</span>", unsafe_allow_html=True)
            if feedback:
                st.caption(f"💡 {', '.join(feedback)}")

        confirm_password = st.text_input("Confirm Password", type="password")
        question = st.selectbox("Security Question", SECURITY_QUESTIONS)
        answer = st.text_input("Security Answer")

        if st.button("Sign Up"):
            username = username.strip()
            email = email.strip().lower()
            answer_clean = answer.strip().lower()

            errors = []
            if not username:
                errors.append("Username is required")
            if not is_valid_email(email):
                errors.append("Valid email is required")
            if not password:
                errors.append("Password is required")
            else:
                strength, feedback = check_password_strength(password)
                if strength == "Weak":
                    errors.append(f"Password too weak: {', '.join(feedback)}")
            if password != confirm_password:
                errors.append("Passwords do not match")
            if not answer_clean:
                errors.append("Security answer is required")

            if errors:
                for e in errors:
                    st.error(e)
                return

            if check_user_exists_by_email(email):
                st.error("Email already registered")
                return
            if check_user_exists_by_username(username):
                st.error("Username already taken")
                return

            ok, msg = register_user(username, email, password, question, answer_clean)
            if ok:
                st.success("✅ Account created successfully!")
                time.sleep(1)
                st.session_state["page"] = "login"
                st.rerun()
            else:
                st.error(f"Registration error: {msg}")

        if st.button("← Back to Login"):
            st.session_state["page"] = "login"
            st.rerun()

# ==============================
# LOGIN PAGE
# ==============================

def login_page():
    col1, col2, col3 = st.columns([1, 2, 1])
    with col2:
        st.title("🔐 Login")

        email = st.text_input("Email")
        password = st.text_input("Password", type="password")

        if st.button("Login"):
            if not email or not password:
                st.error("All fields are required")
                return

            is_locked, wait_time = is_rate_limited(email)
            if is_locked:
                st.error(f"⛔ Account locked! Too many failed attempts. Try again in {wait_time} seconds.")
                return

            success, username_db = authenticate_user(email, password)
            if success:
                token = create_access_token({"sub": email, "username": username_db})
                st.session_state["jwt_token"] = token
                st.session_state["username"] = username_db
                st.success(f"✅ Welcome back, {username_db}!")
                time.sleep(1)
                st.rerun()
            else:
                st.error("Invalid email or password")
                old_dt = check_is_old_password(email, password)
                if old_dt:
                    st.warning(f"⚠️ Note: This was a previously used password ({get_relative_time(old_dt)})")
                attempts, _ = get_login_attempts(email)
                remaining = MAX_LOGIN_ATTEMPTS - attempts
                if remaining > 0:
                    st.caption(f"⚠️ {remaining} attempt(s) remaining before lockout")

        col_a, col_b = st.columns(2)
        with col_a:
            if st.button("Create Account"):
                st.session_state["page"] = "signup"
                st.rerun()
        with col_b:
            if st.button("Forgot Password"):
                st.session_state["page"] = "forgot"
                st.rerun()

# ==============================
# FORGOT PASSWORD PAGE
# ==============================

def forgot_password_page():
    col1, col2, col3 = st.columns([1, 2, 1])
    with col2:
        st.title("🔐 Reset Password")

        # Stage management
        if "forgot_stage" not in st.session_state:
            st.session_state["forgot_stage"] = "email"

        stage = st.session_state["forgot_stage"]

        # --- STAGE 1: Enter Email ---
        if stage == "email":
            st.markdown("##### Step 1: Enter your registered email")
            email_input = st.text_input("Registered Email")
            if st.button("Next →"):
                if not is_valid_email(email_input):
                    st.error("Enter a valid email")
                elif not check_user_exists_by_email(email_input):
                    st.error("Email not found")
                else:
                    st.session_state["reset_email"] = email_input.strip().lower()
                    st.session_state["forgot_stage"] = "security"
                    st.rerun()

        # --- STAGE 2: Security Question ---
        elif stage == "security":
            st.markdown("##### Step 2: Answer your security question")
            email = st.session_state["reset_email"]
            result = get_security_question(email)
            if not result:
                st.error("Could not retrieve security question")
                return
            question, correct_answer = result
            st.info(f"🔒 {question}")
            user_answer = st.text_input("Your Answer")
            if st.button("Verify Answer"):
                if user_answer.strip().lower() == correct_answer:
                    st.session_state["forgot_stage"] = "otp_send"
                    st.rerun()
                else:
                    st.error("Incorrect answer. Please try again.")

        # --- STAGE 3: Send OTP ---
        elif stage == "otp_send":
            st.markdown("##### Step 3: Send OTP to your email")
            email = st.session_state["reset_email"]
            st.success(f"✅ Security question verified!")
            st.info(f"We'll send an OTP to: **{email}**")
            if st.button("📧 Send OTP"):
                otp = generate_otp()
                ok, msg = send_otp_email(email, otp)
                if ok:
                    st.session_state["otp_token"] = create_otp_token(otp, email)
                    st.session_state["forgot_stage"] = "otp_verify"
                    st.success("OTP sent! Check your email.")
                    time.sleep(1)
                    st.rerun()
                else:
                    st.error(f"Failed to send OTP: {msg}")
                    st.caption("Tip: Make sure EMAIL_PASSWORD env variable is set correctly.")

        # --- STAGE 4: Verify OTP ---
        elif stage == "otp_verify":
            st.markdown("##### Step 4: Enter the OTP from your email")
            otp_input = st.text_input("Enter 6-digit OTP", max_chars=6)
            email = st.session_state["reset_email"]
            if st.button("Verify OTP"):
                ok, msg = verify_otp_token(st.session_state.get("otp_token", ""), otp_input, email)
                if ok:
                    st.session_state["forgot_stage"] = "reset"
                    st.rerun()
                else:
                    st.error(f"OTP verification failed: {msg}")
            if st.button("🔄 Resend OTP"):
                st.session_state["forgot_stage"] = "otp_send"
                st.rerun()

        # --- STAGE 5: Reset Password ---
        elif stage == "reset":
            st.markdown("##### Step 5: Set your new password")
            email = st.session_state["reset_email"]
            new_password = st.text_input("New Password", type="password")
            if new_password:
                strength, feedback = check_password_strength(new_password)
                color = {"Strong": "#bf00ff", "Medium": "#ffa500", "Weak": "#ff4b4b"}[strength]
                st.markdown(f"Strength: <span style='color:{color};font-weight:bold;'>{strength}</span>", unsafe_allow_html=True)
                if feedback:
                    st.caption(f"💡 {', '.join(feedback)}")
            confirm_password = st.text_input("Confirm New Password", type="password")

            if st.button("Update Password"):
                if new_password != confirm_password:
                    st.error("Passwords do not match")
                else:
                    strength, feedback = check_password_strength(new_password)
                    if strength == "Weak":
                        st.error(f"Password too weak: {', '.join(feedback)}")
                    elif check_password_reused(email, new_password):
                        st.error("⚠️ Cannot reuse a previous password. Please choose a new one.")
                    else:
                        update_password(email, new_password)
                        st.success("✅ Password updated successfully!")
                        # Clear reset state
                        for key in ["forgot_stage", "reset_email", "otp_token"]:
                            st.session_state.pop(key, None)
                        time.sleep(1)
                        st.session_state["page"] = "login"
                        st.rerun()

        st.markdown("---")
        if st.button("← Back to Login"):
            for key in ["forgot_stage", "reset_email", "otp_token"]:
                st.session_state.pop(key, None)
            st.session_state["page"] = "login"
            st.rerun()

# ==============================
# GAUGE HELPER
# ==============================

def create_gauge(value, title, min_val=0, max_val=100, color="#bf00ff"):
    fig = go.Figure(go.Indicator(
        mode="gauge+number",
        value=value,
        title={'text': title, 'font': {'color': color, 'size': 14}},
        number={'font': {'color': color, 'size': 20}},
        gauge={
            'axis': {'range': [min_val, max_val], 'tickwidth': 1, 'tickcolor': color},
            'bar': {'color': color},
            'bgcolor': "#1f2937",
            'borderwidth': 2,
            'bordercolor': "#374151",
            'steps': [{'range': [min_val, max_val], 'color': "#0e1117"}],
        }
    ))
    fig.update_layout(
        paper_bgcolor="#0e1117",
        font={'color': "#ffffff", 'family': "Courier New"},
        height=250,
        margin=dict(l=10, r=10, t=40, b=10)
    )
    return fig

# ==============================
# CHAT PAGE
# ==============================

def chat_page(username):
    st.title("🤖 Infosys LLM Chat")
    if "messages" not in st.session_state:
        st.session_state["messages"] = []

    for msg in st.session_state["messages"]:
        with st.chat_message(msg["role"]):
            st.markdown(msg["content"])

    if prompt := st.chat_input("Ask me anything..."):
        st.session_state["messages"].append({"role": "user", "content": prompt})
        with st.chat_message("user"):
            st.markdown(prompt)
        with st.chat_message("assistant"):
            response = f"Hello {username}! You said: *{prompt}* — (This is a secure mock response)"
            st.markdown(response)
            st.session_state["messages"].append({"role": "assistant", "content": response})

# ==============================
# READABILITY PAGE
# ==============================

def readability_page():
    st.title("📖 Text Readability Analyzer")

    tab1, tab2 = st.tabs(["✍️ Input Text", "📂 Upload File (TXT/PDF)"])
    text_input = ""

    with tab1:
        raw_text = st.text_area("Enter text to analyze (min 50 chars):", height=200)
        if raw_text:
            text_input = raw_text

    with tab2:
        try:
            import PyPDF2
            uploaded_file = st.file_uploader("Upload a file", type=["txt", "pdf"])
            if uploaded_file:
                if uploaded_file.type == "application/pdf":
                    reader = PyPDF2.PdfReader(uploaded_file)
                    text = ""
                    for page in reader.pages:
                        text += page.extract_text() + "\n"
                    text_input = text
                    st.info(f"✅ Loaded {len(reader.pages)} pages from PDF.")
                else:
                    text_input = uploaded_file.read().decode("utf-8")
                    st.info(f"✅ Loaded TXT file: {uploaded_file.name}")
        except Exception as e:
            st.error(f"Error reading file: {e}")

    if st.button("Analyze Readability", type="primary"):
        if len(text_input) < 50:
            st.error("Text is too short (min 50 chars). Please enter more text.")
        else:
            with st.spinner("Calculating metrics..."):
                analyzer = ReadabilityAnalyzer(text_input)
                scores   = analyzer.get_all_metrics()
                num_sentences = analyzer.num_sentences
                num_words     = analyzer.num_words
                num_syllables = analyzer.num_syllables
                complex_words = analyzer.complex_words
                char_count    = analyzer.char_count

            st.markdown("---")
            st.subheader("📊 Analysis Results")

            avg_grade = (scores['Flesch-Kincaid Grade'] + scores['Gunning Fog'] +
                         scores['SMOG Index'] + scores['Coleman-Liau']) / 4

            if avg_grade <= 6:   level, lcolor = "Beginner (Elementary)",            "#28a745"
            elif avg_grade <= 10: level, lcolor = "Intermediate (Middle School)",     "#17a2b8"
            elif avg_grade <= 14: level, lcolor = "Advanced (High School/College)",   "#ffc107"
            else:                 level, lcolor = "Expert (Professional/Academic)",   "#dc3545"

            st.markdown(f"""
            <div style="background-color:#1f2937;padding:20px;border-radius:10px;
                        border-left:5px solid {lcolor};text-align:center;">
                <h2 style="margin:0;color:{lcolor} !important;">Overall Level: {level}</h2>
                <p style="margin:5px 0 0 0;color:#9ca3af;">Approximate Grade Level: {int(avg_grade)}</p>
            </div>
            """, unsafe_allow_html=True)

            st.markdown("### 📈 Detailed Metrics")

            c1, c2, c3 = st.columns(3)
            with c1:
                st.plotly_chart(create_gauge(scores["Flesch Reading Ease"], "Flesch Reading Ease", 0, 100, "#bf00ff"), use_container_width=True)
                with st.expander("ℹ️ About Flesch Ease"):
                    st.caption("0-100 Scale. Higher is easier. 60-70 is standard.")
            with c2:
                st.plotly_chart(create_gauge(scores["Flesch-Kincaid Grade"], "Flesch-Kincaid Grade", 0, 20, "#ff00ff"), use_container_width=True)
                with st.expander("ℹ️ About Kincaid Grade"):
                    st.caption("US Grade Level. 8.0 means 8th grader can understand.")
            with c3:
                st.plotly_chart(create_gauge(scores["SMOG Index"], "SMOG Index", 0, 20, "#ffff00"), use_container_width=True)
                with st.expander("ℹ️ About SMOG"):
                    st.caption("Commonly used for medical writing. Based on polysyllables.")

            c4, c5 = st.columns(2)
            with c4:
                st.plotly_chart(create_gauge(scores["Gunning Fog"], "Gunning Fog", 0, 20, "#00ccff"), use_container_width=True)
                with st.expander("ℹ️ About Gunning Fog"):
                    st.caption("Based on sentence length and complex words.")
            with c5:
                st.plotly_chart(create_gauge(scores["Coleman-Liau"], "Coleman-Liau", 0, 20, "#ff9900"), use_container_width=True)
                with st.expander("ℹ️ About Coleman-Liau"):
                    st.caption("Based on characters instead of syllables.")

            st.markdown("### 📝 Text Statistics")
            s1, s2, s3, s4, s5 = st.columns(5)
            s1.metric("Sentences",    num_sentences)
            s2.metric("Words",        num_words)
            s3.metric("Syllables",    num_syllables)
            s4.metric("Complex Words", complex_words)
            s5.metric("Characters",   char_count)

            st.markdown("### 📊 Score Comparison Chart")
            metric_names  = list(scores.keys())
            metric_values = list(scores.values())
            colors = ["#bf00ff","#ff00ff","#ffff00","#00ccff","#ff9900"]
            bar_fig = go.Figure(go.Bar(
                x=metric_names, y=metric_values,
                marker_color=colors,
                text=[f"{v:.1f}" for v in metric_values],
                textposition="outside",
                textfont=dict(color="#ffffff", size=13)
            ))
            bar_fig.update_layout(
                paper_bgcolor="#0e1117", plot_bgcolor="#1f2937",
                font=dict(color="#ffffff", family="Courier New"),
                xaxis=dict(tickfont=dict(color="#bf00ff", size=12), gridcolor="#374151"),
                yaxis=dict(tickfont=dict(color="#bf00ff"), gridcolor="#374151"),
                title=dict(text="Readability Scores at a Glance",
                           font=dict(color="#bf00ff", size=16), x=0.5),
                margin=dict(l=20, r=20, t=60, b=20), height=380
            )
            st.plotly_chart(bar_fig, use_container_width=True)

            st.markdown("### 📋 Score Summary Table")
            import pandas as pd
            grade_metrics = {k: v for k, v in scores.items() if k != "Flesch Reading Ease"}
            table_data = {
                "Metric": list(scores.keys()),
                "Score":  [round(v, 2) for v in scores.values()],
                "Interpretation": [
                    f"{'Easy' if scores['Flesch Reading Ease']>=60 else 'Hard'} to read",
                    f"Grade {int(scores['Flesch-Kincaid Grade'])} level",
                    f"Grade {int(scores['SMOG Index'])} level",
                    f"Grade {int(scores['Gunning Fog'])} level",
                    f"Grade {int(scores['Coleman-Liau'])} level",
                ]
            }
            st.dataframe(pd.DataFrame(table_data), use_container_width=True, hide_index=True)

# ==============================
# ADMIN PAGE
# ==============================

def admin_page():
    payload = verify_token(st.session_state["jwt_token"])
    if not payload:
        st.session_state["jwt_token"] = None
        st.session_state["page"] = "login"
        st.rerun()
        return

    with st.sidebar:
        st.markdown("<h2 style='color:#bf00ff;text-align:center;font-family:Courier New;'>LLM</h2>", unsafe_allow_html=True)
        st.markdown(f"**👤 {payload.get('username', 'Admin')}**")
        st.markdown("---")
        try:
            from streamlit_option_menu import option_menu
            selected = option_menu("Infosys LLM", ["Admin"], icons=["shield-lock"],
                menu_icon="cast", default_index=0,
                styles={
                    "container": {"background-color": "#1a1c24"},
                    "icon": {"color": "#bf00ff"},
                    "nav-link": {"color": "#9ca3af", "font-family": "Courier New"},
                    "nav-link-selected": {"background-color": "#bf00ff", "color": "#0e1117"},
                })
        except:
            st.markdown("**🛡️ Admin Panel**")
        st.markdown("---")
        if st.button("🔓 Log Out"):
            st.session_state["jwt_token"] = None
            st.session_state["page"] = "login"
            st.rerun()

    st.title("🛡️ Admin Panel")
    users = get_all_users()
    st.metric("Total Users", len(users))
    st.markdown("---")

    c1, c2, c3, c4 = st.columns([2, 3, 2, 1])
    c1.markdown("**Username**"); c2.markdown("**Email**")
    c3.markdown("**Joined**");   c4.markdown("**Action**")
    st.markdown("---")

    for uname, uemail, ucreated in users:
        c1, c2, c3, c4 = st.columns([2, 3, 2, 1])
        c1.write(f"**{uname}**")
        c2.write(uemail)
        c3.write(get_relative_time(ucreated) if ucreated else "N/A")
        if uname.lower() != "admin":
            if c4.button("🗑️", key=f"del_{uemail}", help=f"Delete {uemail}"):
                delete_user(uemail)
                st.warning(f"Deleted {uemail}")
                time.sleep(0.5)
                st.rerun()

# ==============================
# DASHBOARD (with sidebar nav)
# ==============================

def dashboard_page():
    token = st.session_state["jwt_token"]
    payload = verify_token(token)

    if not payload:
        st.session_state["jwt_token"] = None
        st.session_state["page"] = "login"
        st.rerun()
        return

    username = payload.get("username", "User")
    email    = payload.get("sub", "")

    # Init nav state
    if "nav_selected" not in st.session_state:
        st.session_state["nav_selected"] = "Chat"

    with st.sidebar:
        st.markdown("<h2 style='color:#bf00ff;text-align:center;font-family:Courier New;'>🔷 Infosys</h2>", unsafe_allow_html=True)
        st.markdown(f"### 👤 {username}")
        st.markdown(f"<small>{email}</small>", unsafe_allow_html=True)
        st.markdown("---")
        st.markdown("#### Navigation")

        if st.button("💬  Chat", use_container_width=True,
                     type="primary" if st.session_state["nav_selected"] == "Chat" else "secondary"):
            st.session_state["nav_selected"] = "Chat"
            st.rerun()

        if st.button("📖  Readability", use_container_width=True,
                     type="primary" if st.session_state["nav_selected"] == "Readability" else "secondary"):
            st.session_state["nav_selected"] = "Readability"
            st.rerun()

        st.markdown("---")
        if st.button("🔓  Log Out", use_container_width=True):
            st.session_state["jwt_token"] = None
            st.session_state["username"] = None
            st.session_state["nav_selected"] = "Chat"
            st.session_state["page"] = "login"
            st.rerun()

    selected = st.session_state["nav_selected"]
    if selected == "Chat":
        chat_page(username)
    elif selected == "Readability":
        readability_page()

# ==============================
# ROUTER
# ==============================

if st.session_state.get("jwt_token"):
    payload = verify_token(st.session_state["jwt_token"])
    if payload:
        username = payload.get("username", "")
        if username.lower() == "admin":
            admin_page()
        else:
            dashboard_page()
    else:
        st.session_state["jwt_token"] = None
        st.session_state["page"] = "login"
        st.rerun()
else:
    page = st.session_state.get("page", "login")
    if page == "signup":
        signup_page()
    elif page == "forgot":
        forgot_password_page()
    else:
        login_page()



Writing app.py


In [ ]:

import os, subprocess, time
from pyngrok import ngrok
from google.colab import userdata

os.environ['EMAIL_PASSWORD'] = userdata.get('EMAIL_PASSWORD')
os.environ['EMAIL_ID']  = userdata.get('EMAIL_ID')
os.environ['JWT_SECRET']     = 'super-secret-change-me'
os.environ['DB_NAME']        = 'milestone1_users'
os.environ['DB_USER']        = 'postgres'
os.environ['DB_PASSWORD']    = 'postgres'
os.environ['DB_HOST']        = 'localhost'
os.environ['DB_PORT']        = '5432'

ngrok.set_auth_token(userdata.get('NGROK_AUTHTOKEN'))
ngrok.kill()
time.sleep(1)

st_proc = subprocess.Popen(
    ['streamlit', 'run', 'app.py', '--server.port', '8501', '--server.headless', 'true'],
    env=os.environ.copy()
)
time.sleep(4)

public_url = ngrok.connect(8501).public_url
print('🚀 App running at:', public_url)

input('Press ENTER to stop...')
st_proc.terminate(); ngrok.kill()


🚀 App running at: https://unawake-cirrosely-jaimee.ngrok-free.dev
